# Demo C1 — Final Project: Document Research Agent

Capstone project building on `Demo_C1.ipynb` — instead of separate feature demos, this notebook composes several of them into one working agent, built with the current (v1) LangChain/LangGraph APIs. See `CLAUDE.md` at the repo root for the version-specific gotchas already worked out (`create_agent` shape, the `chromadb` pin, `langchain_classic`, etc.).

**Roadmap** — built one stage at a time, each verified before moving to the next:

1. **Knowledge base** — load a PDF, split it, embed it, build a retriever. Tested standalone.
2. **`search_documents` tool** — wrap the retriever as an agent tool. Tested standalone.
3. **Compose tool** — a multi-step LCEL chain (the `RunnablePassthrough.assign` pattern from `Demo_C1`) exposed as a second tool.
4. **Agent, one tool** — wire `search_documents` into `create_agent`.
5. **Agent, both tools** — add the compose tool.
6. **Memory** — add a `checkpointer` so the agent remembers earlier turns.
7. **Structured output** — add a `response_format` Pydantic model.

Only stage 1 is scaffolded below. Later stages get added once this one is working.

## Setup

In [1]:
import os
import warnings

import pandas as pd
from dotenv import load_dotenv
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_classic.storage import InMemoryStore
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import CharacterTextSplitter

warnings.filterwarnings("ignore")
load_dotenv()

/var/folders/pf/7lwsqjw92g96dl5sfdckf9_r0000gn/T/ipykernel_17648/2781504271.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


True

In [2]:
from IPython.core.interactiveshell import InteractiveShell

InteractiveShell.ast_node_interactivity = "all"

In [3]:
model = ChatOpenAI(model="gpt-4o-mini", api_key=os.environ.get("OPENAI_API_KEY"))
embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-small", api_key=os.environ.get("OPENAI_API_KEY")
)

PDF_PATH = "../../data/O-príncipe-Nicolau-Maquiavel.pdf"  # same source Demo_C1 already loads
CHROMA_PERSIST_DIR = (
    "../../data/chroma_demo_c1_project"  # persisted so we don't re-embed every restart
)

## 1. Knowledge base

Build the retrieval pipeline from `Demo_C1`'s DOCUMENT / TEXT SPLITTERS / EMBEDDINGS / VECTOR STORES / RETRIEVERS sections, combined into one `ParentDocumentRetriever` (imported from `langchain_classic.retrievers` — see `CLAUDE.md` for why).

You already built this exact pattern in `Demo_C1.ipynb`'s **PARENT DOCUMENT RETRIEVER** section — that's your reference, not this notebook.

Once it runs: test with `retriever.invoke("some query")` and actually look at what comes back — is it a small chunk or the full parent document? That answer matters once this becomes a tool the agent calls, so don't skip inspecting it.

In [ ]:
# vector_store.delete_collection()

In [5]:
# TODO: load the PDF at PDF_PATH
# HINT: PyPDFLoader(...).load() returns a list[Document]
pdf_docs = PyPDFLoader(PDF_PATH)
pdf_loader = pdf_docs.load()

# TODO: define a child_splitter (small chunks, for the vector index) and a
# parent_splitter (larger chunks, for full context) — ParentDocumentRetriever needs both
child_splitter = CharacterTextSplitter(separator="\n", chunk_size=400, chunk_overlap=50)
parent_splitter = CharacterTextSplitter(separator="\n", chunk_size=4000, chunk_overlap=100)

# TODO: create a persisted Chroma vector store (for child chunks) and an
# InMemoryStore (for parent docs), then assemble the ParentDocumentRetriever
# HINT: from langchain_classic.storage import InMemoryStore
# HINT: from langchain_classic.retrievers import ParentDocumentRetriever
# HINT: Chroma(..., persist_directory=CHROMA_PERSIST_DIR, embedding_function=embedding_model)
vector_store = Chroma(
    collection_name="C1_project",
    persist_directory=CHROMA_PERSIST_DIR,
    embedding_function=embedding_model,
)
docstore = InMemoryStore()  # Your code here
retriever = ParentDocumentRetriever(
    vectorstore=vector_store,
    docstore=docstore,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)

# TODO: index pdf_docs into the retriever
retriever.add_documents(pdf_loader)

# TODO: test it — inspect what actually comes back, not just that it ran
test_results = retriever.invoke("qual o nome do livro?")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


In [6]:
type(test_results)
len(test_results)
type(test_results[0])

list

2

langchain_core.documents.base.Document

In [7]:
len(test_results[0].page_content)
print(test_results[0].page_content)

103

NICOLAU 
MAQU IAVEL
O príncipe
Tradução de
MAURÍCIO SANT ANA DIAS
Prefácio de
FERNANDO HENRIQUE CARDOSO


Once this stage runs and `test_results` looks right to you, let's check in before scaffolding stage 2 (the `search_documents` tool).

### What to measure here ?  
> Did the parent-lookup step fired ?  
>> compare the result against the child:
>>> Close to 400 → you're likely still getting child-level content back, not parents. 
>>>> Meaningfully bigger than 400 → the parent lookup did its job. 

In [8]:
# O resultado tem 103 chars,que é menos do que os 400 do child splitter.
# para saber o que gerou a resposta
child_matches = vector_store.similarity_search("qual o nome do livro?")

In [9]:
type(child_matches)
len(child_matches)
type(child_matches[0])
# child_matches retorna uma lista com 4 Document

list

4

langchain_core.documents.base.Document

In [10]:
# child_matches[i].page_content → the actual child chunk that matched in Chroma (small, ~400 chars)

# child_matches[i].metadata["doc_id"] → the in-memory id, i.e. the exact key test_results[i] was fetched by from docstore

In [11]:
results_metadata = pd.DataFrame(
    [
        {"memory_doc_id": i.metadata["doc_id"], "child_page_content": i.page_content}
        for i in child_matches
    ]
)
results_metadata

,memory_doc_id,child_page_content
0,3cf66a60-3323-4291-aa2b-b74786c377db,NICOLAU \nMAQU IAVEL\nO príncipe\nTradução de\...
1,50495ac7-d0ce-4e18-896e-3c9e516e0d1e,NICOLAU \nMAQU IAVEL\nO príncipe\nTradução de\...
2,23e56b56-05cf-454c-930c-63255e1c3e11,trademarks of Penguin Books Limited and/or Pen...
3,81d929a2-0ad2-4889-b316-33ff9fdd4f06,trademarks of Penguin Books Limited and/or Pen...


In [12]:
_ = results_metadata["memory_doc_id"]
len(_)
parent_page_content = [i.page_content for i in docstore.mget(_) if i is not None]
parent_page_content

4

['NICOLAU \nMAQU IAVEL\nO príncipe\nTradução de\nMAURÍCIO SANT ANA DIAS\nPrefácio de\nFERNANDO HENRIQUE CARDOSO',
 'Copyright das notas © 1961, 1975, 1981, 1995, 1999 by George Bull\nCopyright da introdução © 1999 by Anthony Grafton\nCopyright do prefácio © 2010 by Fernando Henrique Cardoso\nGrafia atualizada segundo o Acordo Ortográfico da Língua Portuguesa de 1990, que entrou\nem vigor no Brasil em 2009.\nPenguin and the associated logo and trade dress are registered and/or unregistered\ntrademarks of Penguin Books Limited and/or Penguin Group ( USA) Inc. Used with\npermission.\nPublished by Companhia das Letras in association with Penguin Group ( USA) Inc.\nTÍTULO ORIGINAL\nDe principatibus\nCAPA E PROJETO GRÁFICO PENGUIN-COMPANHIA\nRaul Loureiro, Cláudia W arrak\nTRADUÇÃO DOS APÊNDICES\nLuiz A. de Araújo\nPREPARAÇÃO\nSilvia Maximini Félix\nREVISÃO\nAna Maria Barbosa\nHuendel V iana\nISBN: 978-85-63397-60-7\nTodos os direitos desta edição reservados à\nEDITORA SCHWARCZ LTDA.\nRua Ba

In [13]:
# Findings:
# 1 - the better result len(test_results[0]) = 103, so its smaller than the 400 chars cap. So, seems parent llokup did not work as intended.
# 2 - the parent chunk from result[0] is the same as its child. again, looks parent retrievering gone wrong.
# 3 - on the 4th result,   63eef901-6ce4-4560-98bb-367887345e81, parent is equal to child also.

In [14]:
# # Answers
# 1 - the result is only 103 chars long because the splitting operated page a page, not in a  concatenated document without page boundaries. This 103 chars text was the content of page 3.
# 2 - The parent-lookup worked as intended. See below, child is within the parent.

In [15]:
len(test_results[1].page_content)
print(test_results[1].page_content)

len(results_metadata["child_page_content"][1])
results_metadata["child_page_content"][1]

len(parent_page_content[1])
parent_page_content[1]

994

Copyright das notas © 1961, 1975, 1981, 1995, 1999 by George Bull
Copyright da introdução © 1999 by Anthony Grafton
Copyright do prefácio © 2010 by Fernando Henrique Cardoso
Grafia atualizada segundo o Acordo Ortográfico da Língua Portuguesa de 1990, que entrou
em vigor no Brasil em 2009.
Penguin and the associated logo and trade dress are registered and/or unregistered
trademarks of Penguin Books Limited and/or Penguin Group ( USA) Inc. Used with
permission.
Published by Companhia das Letras in association with Penguin Group ( USA) Inc.
TÍTULO ORIGINAL
De principatibus
CAPA E PROJETO GRÁFICO PENGUIN-COMPANHIA
Raul Loureiro, Cláudia W arrak
TRADUÇÃO DOS APÊNDICES
Luiz A. de Araújo
PREPARAÇÃO
Silvia Maximini Félix
REVISÃO
Ana Maria Barbosa
Huendel V iana
ISBN: 978-85-63397-60-7
Todos os direitos desta edição reservados à
EDITORA SCHWARCZ LTDA.
Rua Bandeira Paulista, 702, cj. 32
04532-002 — São Paulo — SP
Telefone: (01 1) 3707-3500 Fax: (01 1) 3707-3501
www .penguincompanhia.com.br


103

'NICOLAU \nMAQU IAVEL\nO príncipe\nTradução de\nMAURÍCIO SANT ANA DIAS\nPrefácio de\nFERNANDO HENRIQUE CARDOSO'

994

'Copyright das notas © 1961, 1975, 1981, 1995, 1999 by George Bull\nCopyright da introdução © 1999 by Anthony Grafton\nCopyright do prefácio © 2010 by Fernando Henrique Cardoso\nGrafia atualizada segundo o Acordo Ortográfico da Língua Portuguesa de 1990, que entrou\nem vigor no Brasil em 2009.\nPenguin and the associated logo and trade dress are registered and/or unregistered\ntrademarks of Penguin Books Limited and/or Penguin Group ( USA) Inc. Used with\npermission.\nPublished by Companhia das Letras in association with Penguin Group ( USA) Inc.\nTÍTULO ORIGINAL\nDe principatibus\nCAPA E PROJETO GRÁFICO PENGUIN-COMPANHIA\nRaul Loureiro, Cláudia W arrak\nTRADUÇÃO DOS APÊNDICES\nLuiz A. de Araújo\nPREPARAÇÃO\nSilvia Maximini Félix\nREVISÃO\nAna Maria Barbosa\nHuendel V iana\nISBN: 978-85-63397-60-7\nTodos os direitos desta edição reservados à\nEDITORA SCHWARCZ LTDA.\nRua Bandeira Paulista, 702, cj. 32\n04532-002 — São Paulo — SP\nTelefone: (01 1) 3707-3500 Fax: (01 1) 3707-3501\nwww .pe

### V2
Implementing the pdf concatenation.

In [63]:
# TODO: load the PDF at PDF_PATH
# HINT: PyPDFLoader(...).load() returns a list[Document]
pdf_docs = PyPDFLoader(PDF_PATH)
pdf_loader = pdf_docs.load()
len(pdf_loader)
pdf_loader[0]

154

Document(metadata={'producer': 'calibre (5.39.1) [https://calibre-ebook.com]', 'creator': 'calibre (5.39.1) [https://calibre-ebook.com]', 'creationdate': '2022-03-13T01:35:03+00:00', 'author': 'Nicolau Maquiavel', 'keywords': 'Political Science, History & Theory', 'moddate': '2022-03-29T14:54:02-03:00', 'title': 'O príncipe', 'source': '../../data/O-príncipe-Nicolau-Maquiavel.pdf', 'total_pages': 154, 'page': 0, 'page_label': '1'}, page_content='')

In [34]:
def concat(docs: list[Document]) -> str:
    return "\n".join(i.page_content for i in docs)


pdf_concat = concat(pdf_loader)
len(pdf_concat)
type(pdf_concat)
pdf_concat[:50]

282670

str

'\nO PRÍNCIPE\nNICOLAU MAQUIAVEL nasceu em Florença e'

In [65]:
new_metadata = {k:v for k,v in pdf_loader[0].metadata.items()}
del new_metadata["page"]
del new_metadata["page_label"]
new_metadata

{'producer': 'calibre (5.39.1) [https://calibre-ebook.com]',
 'creator': 'calibre (5.39.1) [https://calibre-ebook.com]',
 'creationdate': '2022-03-13T01:35:03+00:00',
 'author': 'Nicolau Maquiavel',
 'keywords': 'Political Science, History & Theory',
 'moddate': '2022-03-29T14:54:02-03:00',
 'title': 'O príncipe',
 'source': '../../data/O-príncipe-Nicolau-Maquiavel.pdf',
 'total_pages': 154}

In [66]:
pdf_concat_document = Document(page_content=pdf_concat, metadata = new_metadata)
type(pdf_concat_document)

langchain_core.documents.base.Document

In [67]:
# TODO: define a child_splitter (small chunks, for the vector index) and a
# parent_splitter (larger chunks, for full context) — ParentDocumentRetriever needs both
child_splitter = CharacterTextSplitter(separator="\n", chunk_size=400, chunk_overlap=50)
parent_splitter = CharacterTextSplitter(separator="\n", chunk_size=4000, chunk_overlap=100)

# TODO: create a persisted Chroma vector store (for child chunks) and an
# InMemoryStore (for parent docs), then assemble the ParentDocumentRetriever
# HINT: from langchain_classic.storage import InMemoryStore
# HINT: from langchain_classic.retrievers import ParentDocumentRetriever
# HINT: Chroma(..., persist_directory=CHROMA_PERSIST_DIR, embedding_function=embedding_model)
vector_store = Chroma(
    collection_name="C1_project_v2",
    persist_directory=CHROMA_PERSIST_DIR,
    embedding_function=embedding_model,
)
docstore = InMemoryStore()  # Your code here
retriever = ParentDocumentRetriever(
    vectorstore=vector_store,
    docstore=docstore,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)

# TODO: index pdf_docs into the retriever
retriever.add_documents([pdf_concat_document])

# TODO: test it — inspect what actually comes back, not just that it ran
test_results = retriever.invoke("qual o nome do livro?")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


In [68]:
child_matches = vector_store.similarity_search("qual o nome do livro?")

In [75]:
type(child_matches)
len(child_matches)
type(child_matches[0])
len(child_matches[0].page_content)
child_matches[0].page_content

list

4

langchain_core.documents.base.Document

390

'trademarks of Penguin Books Limited and/or Penguin Group ( USA) Inc. Used with\npermission.\nPublished by Companhia das Letras in association with Penguin Group ( USA) Inc.\nTÍTULO ORIGINAL\nDe principatibus\nCAPA E PROJETO GRÁFICO PENGUIN-COMPANHIA\nRaul Loureiro, Cláudia W arrak\nTRADUÇÃO DOS APÊNDICES\nLuiz A. de Araújo\nPREPARAÇÃO\nSilvia Maximini Félix\nREVISÃO\nAna Maria Barbosa\nHuendel V iana'

In [70]:
results_metadata =  pd.DataFrame([{"doc_id": i.metadata["doc_id"], "child_page_content": i.page_content} for i in child_matches])
results_metadata

,doc_id,child_page_content
0,c71b65db-543c-4f75-adfb-c645b4e1ccc9,trademarks of Penguin Books Limited and/or Pen...
1,c71b65db-543c-4f75-adfb-c645b4e1ccc9,REVISÃO\nAna Maria Barbosa\nHuendel V iana\nIS...
2,fc07e740-090f-4d30-be9a-d2b67115d21d,XXV. Em que medida a fortuna controla as coisa...
3,c71b65db-543c-4f75-adfb-c645b4e1ccc9,"Copyright das notas © 1961, 1975, 1981, 1995, ..."


In [71]:
results_metadata["child_page_content"][0]

'trademarks of Penguin Books Limited and/or Penguin Group ( USA) Inc. Used with\npermission.\nPublished by Companhia das Letras in association with Penguin Group ( USA) Inc.\nTÍTULO ORIGINAL\nDe principatibus\nCAPA E PROJETO GRÁFICO PENGUIN-COMPANHIA\nRaul Loureiro, Cláudia W arrak\nTRADUÇÃO DOS APÊNDICES\nLuiz A. de Araújo\nPREPARAÇÃO\nSilvia Maximini Félix\nREVISÃO\nAna Maria Barbosa\nHuendel V iana'

'JOLY, M. Diálogo no inferno entre Maquiavel e Montesquieu , Unesp, 2009.\nSKINNER, Q. As fundações do pensamento político moderno , São Paulo, Companhia das\nLetras, 1989.\nVIROLI, M. O sorriso de Nicolau: história de Maquiavel , São Paulo, Estação Liberdade, 2002.\nWHITE, M. Maquiavel, um homem incompreendido , Rio de Janeiro, Record, 2007.\nCopyright das notas © 1961, 1975, 1981, 1995, 1999 by George Bull\nCopyright da introdução © 1999 by Anthony Grafton\nCopyright do prefácio © 2010 by Fernando Henrique Cardoso\nGrafia atualizada segundo o Acordo Ortográfico da Língua Portuguesa de 1990, que entrou\nem vigor no Brasil em 2009.\nPenguin and the associated logo and trade dress are registered and/or unregistered\ntrademarks of Penguin Books Limited and/or Penguin Group ( USA) Inc. Used with\npermission.\nPublished by Companhia das Letras in association with Penguin Group ( USA) Inc.\nTÍTULO ORIGINAL\nDe principatibus\nCAPA E PROJETO GRÁFICO PENGUIN-COMPANHIA\nRaul Loureiro, Cláudia W

In [ ]:
_

0    NICOLAU \nMAQU IAVEL\nO príncipe\nTradução de\...
1    trademarks of Penguin Books Limited and/or Pen...
2    REVISÃO\nAna Maria Barbosa\nHuendel V iana\nIS...
3    Cronologia\nGlossário de nomes próprios\nOutra...
Name: child_page_content, dtype: str

In [81]:
len(test_results[0].page_content)
test_results[0].page_content

len(child_matches[0].page_content)
child_matches[0].page_content

len(docstore.mget(["c71b65db-543c-4f75-adfb-c645b4e1ccc9"])[0].page_content)
docstore.mget(["c71b65db-543c-4f75-adfb-c645b4e1ccc9"])[0].page_content

1334

'JOLY, M. Diálogo no inferno entre Maquiavel e Montesquieu , Unesp, 2009.\nSKINNER, Q. As fundações do pensamento político moderno , São Paulo, Companhia das\nLetras, 1989.\nVIROLI, M. O sorriso de Nicolau: história de Maquiavel , São Paulo, Estação Liberdade, 2002.\nWHITE, M. Maquiavel, um homem incompreendido , Rio de Janeiro, Record, 2007.\nCopyright das notas © 1961, 1975, 1981, 1995, 1999 by George Bull\nCopyright da introdução © 1999 by Anthony Grafton\nCopyright do prefácio © 2010 by Fernando Henrique Cardoso\nGrafia atualizada segundo o Acordo Ortográfico da Língua Portuguesa de 1990, que entrou\nem vigor no Brasil em 2009.\nPenguin and the associated logo and trade dress are registered and/or unregistered\ntrademarks of Penguin Books Limited and/or Penguin Group ( USA) Inc. Used with\npermission.\nPublished by Companhia das Letras in association with Penguin Group ( USA) Inc.\nTÍTULO ORIGINAL\nDe principatibus\nCAPA E PROJETO GRÁFICO PENGUIN-COMPANHIA\nRaul Loureiro, Cláudia W

390

'trademarks of Penguin Books Limited and/or Penguin Group ( USA) Inc. Used with\npermission.\nPublished by Companhia das Letras in association with Penguin Group ( USA) Inc.\nTÍTULO ORIGINAL\nDe principatibus\nCAPA E PROJETO GRÁFICO PENGUIN-COMPANHIA\nRaul Loureiro, Cláudia W arrak\nTRADUÇÃO DOS APÊNDICES\nLuiz A. de Araújo\nPREPARAÇÃO\nSilvia Maximini Félix\nREVISÃO\nAna Maria Barbosa\nHuendel V iana'

1334

'JOLY, M. Diálogo no inferno entre Maquiavel e Montesquieu , Unesp, 2009.\nSKINNER, Q. As fundações do pensamento político moderno , São Paulo, Companhia das\nLetras, 1989.\nVIROLI, M. O sorriso de Nicolau: história de Maquiavel , São Paulo, Estação Liberdade, 2002.\nWHITE, M. Maquiavel, um homem incompreendido , Rio de Janeiro, Record, 2007.\nCopyright das notas © 1961, 1975, 1981, 1995, 1999 by George Bull\nCopyright da introdução © 1999 by Anthony Grafton\nCopyright do prefácio © 2010 by Fernando Henrique Cardoso\nGrafia atualizada segundo o Acordo Ortográfico da Língua Portuguesa de 1990, que entrou\nem vigor no Brasil em 2009.\nPenguin and the associated logo and trade dress are registered and/or unregistered\ntrademarks of Penguin Books Limited and/or Penguin Group ( USA) Inc. Used with\npermission.\nPublished by Companhia das Letras in association with Penguin Group ( USA) Inc.\nTÍTULO ORIGINAL\nDe principatibus\nCAPA E PROJETO GRÁFICO PENGUIN-COMPANHIA\nRaul Loureiro, Cláudia W

## 2. `search_documents` tool

Wrap `retriever` (the V2 one, built on `pdf_concat_document`) as an agent tool, using the `@tool` decorator pattern from `Demo_C1`'s **Custom tool via the `@tool` decorator** section — that's your reference, not this notebook.

A few things to think about before you write it:

- The function needs a type-annotated argument (the search query) and a docstring — `@tool` builds the tool's schema and description from those, so the docstring is what the agent reads to decide when to call this.
- `retriever.invoke(query)` returns `list[Document]`. Tools generally return strings back to the agent — so this function has to reduce that list down to something text-based. What's lost if you just join `page_content` with no separator? What do you gain by including each source's metadata alongside its content?
- Per `CLAUDE.md`: tool `name` must match `^[a-zA-Z0-9_-]+$` — no spaces. `@tool` uses the function name by default, so name the function accordingly.

Test it standalone by calling the function directly (not through an agent) before moving on — same rule as stage 1.

In [ ]:
# TODO: import the @tool decorator
# HINT: from langchain_core.tools import tool

# TODO: define search_documents(query: str) -> str
# - call retriever.invoke(query)
# - reduce the resulting list[Document] into a single string to return
# - write a docstring the agent will use to decide when to call this tool


In [ ]:
# TODO: test standalone — call the underlying function directly (e.g. search_documents.invoke("...")
# or search_documents.func("...") depending on how you call a @tool-wrapped function) and read the output
